# Tools in LlamaIndex

# Using Tools in LlamaIndex

**Defining a clear set of Tools is crucial to performance.** As we discussed in [unit 1](../../unit1/tools), clear tool interfaces are easier for LLMs to use.
Much like a software API interface for human engineers, they can get more out of the tool if it's easy to understand how it works.

There are **four main types of tools in LlamaIndex**:

![Tools](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/tools.png)

1. `FunctionTool`: Convert any Python function into a tool that an agent can use. It automatically figures out how the function works.
2. `QueryEngineTool`: A tool that lets agents use query engines. Since agents are built on query engines, they can also use other agents as tools.
3. `Toolspecs`: Sets of tools created by the community, which often include tools for specific services like Gmail.
4. `Utility Tools`: Special tools that help handle large amounts of data from other tools.



## Let's install the dependencies¶
We will install the dependencies for this unit.

In [2]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface llama-index-tools-google

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 755.4 kB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [llama-index-tools-google]ndex-tools-google]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv(dotenv_path="../.env", override=True)
token = os.environ.get("HF_TOKEN")

# Validate before logging in
try:
    login(token=token, add_to_git_credential=False)
    user = whoami()
    print(f"✅ Logged in as: {user['name']}")
except Exception as e:
    print(f"❌ Login failed: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in as: zaidalissa


We will go over each of them in more detail below.

## Creating a FunctionTool

> [!TIP]
> You can follow the code in this notebook that you can run using Google Colab.

A FunctionTool provides a simple way to wrap any Python function and make it available to an agent.
You can pass either a synchronous or asynchronous function to the tool, along with optional `name` and `description` parameters.
The name and description are particularly important as they help the agent understand when and how to use the tool effectively.
Let's look at how to create a FunctionTool below and then call it.



> [!TIP]
> When using an agent or LLM with function calling, the tool selected (and the arguments written for that tool) rely strongly on the tool name and description of the purpose and arguments of the tool. Learn more about function calling in the Function Calling Guide.



In [3]:
from llama_index.core.tools import FunctionTool

def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"


tool = FunctionTool.from_defaults(
    get_weather,
    name="my_weather_tool",
    description="Useful for getting the weather for a given location.",
)
tool.call("London")

Getting weather for London


ToolOutput(blocks=[TextBlock(block_type='text', text='The weather in London is sunny')], tool_name='my_weather_tool', raw_input={'args': ('London',), 'kwargs': {}}, raw_output='The weather in London is sunny', is_error=False)

## Creating a QueryEngineTool

The `QueryEngine` we defined in the previous unit can be easily transformed into a tool using the `QueryEngineTool` class.
Let's see how to create a `QueryEngineTool` from a `QueryEngine` in the example below.

In [4]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
# llm = HuggingFaceInferenceAPI(model_name="meta-llama/Llama-3.2-3B-Instruct")
llm = HuggingFaceInferenceAPI(model_name="meta-llama/Meta-Llama-3-8B-Instruct")
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="some useful name",
    description="some useful description",
)


In [5]:
response = await tool.acall(
    "Responds about research on the impact of AI on the future of work and society?"
)

In [6]:
response.content

"It's essential to consider the potential consequences of AI on various aspects of society, including employment and social dynamics. As AI continues to advance, it may lead to significant changes in the job market, potentially displacing certain roles while creating new ones. This shift could have far-reaching effects on individuals, communities, and the economy as a whole."

In [7]:
response.raw_output.metadata

{'dc7de9bf-96de-4c87-950f-5598387452bf': {'file_path': '/Users/clean_zaid/Documents/Home/Backups/BOOKS/Trillionaire/Python/jupyter-notebooks/huggingface-ai-agents-course/unit_2.2/data/persona_10.txt',
  'file_name': 'persona_10.txt',
  'file_type': 'text/plain',
  'file_size': 207,
  'creation_date': '2026-04-09',
  'last_modified_date': '2026-04-09'},
 'e02fc50f-907c-4ac7-a542-a53b28e93159': {'file_path': '/Users/clean_zaid/Documents/Home/Backups/BOOKS/Trillionaire/Python/jupyter-notebooks/huggingface-ai-agents-course/unit_2.2/data/persona_10.txt',
  'file_name': 'persona_10.txt',
  'file_type': 'text/plain',
  'file_size': 207,
  'creation_date': '2026-04-09',
  'last_modified_date': '2026-04-10'}}

## Creating Toolspecs

Think of `ToolSpecs` as collections of tools that work together harmoniously - like a well-organized professional toolkit.
Just as a mechanic's toolkit contains complementary tools that work together for vehicle repairs, a `ToolSpec` combines related tools for specific purposes.
For example, an accounting agent's `ToolSpec` might elegantly integrate spreadsheet capabilities, email functionality, and calculation tools to handle financial tasks with precision and efficiency.

And now we can load the toolspec and convert it to a list of tools.

In [8]:
from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec()
tool_spec_list = tool_spec.to_tool_list()
tool_spec_list

To get a more detailed view of the tools, we can take a look at the `metadata` of each tool.

In [ ]:
[print(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]

### Model Context Protocol (MCP) in LlamaIndex

LlamaIndex also allows using MCP tools through a [ToolSpec on the LlamaHub](https://llamahub.ai/l/tools/llama-index-tools-mcp?from=).
You can simply run an MCP server and start using it through the following implementation.

If you want to dive deeper about MCP, you can check our [free MCP Course](https://huggingface.co/learn/mcp-course/). 

In [31]:
!pip install llama-index-tools-mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [llama-index-tools-mcp]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [36]:
# We consider there is a mcp server running on 127.0.0.1:8000, or you can use the mcp client to connect to your own mcp server.
mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
mcp_tool = McpToolSpec(client=mcp_client)

# get the agent
agent = await get_agent(mcp_tool)

# create the agent context
agent_context = Context(agent)

NameError: name 'get_agent' is not defined

In [10]:
from llama_index.core.agent import AgentWorkflow
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
mcp_tool = McpToolSpec(client=mcp_client)
tools = await mcp_tool.to_tool_list_async()

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=tools,
    llm=llm,
    verbose=True,
)
response = await agent.run("What is 25 + 17?")
print(str(response))

[tick] add: AgentWorkflowStartEvent(user_msg='What is 25 + 17?', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is 25 + 17?')])], current_agent_name='Agent')
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is 25 + 17?')])], current_agent_name='Agent')
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Thought: The current language of the user is: English. I...
[p

## Utility Tools

Oftentimes, directly querying an API **can return an excessive amount of data**, some of which may be irrelevant, overflow the context window of the LLM, or unnecessarily increase the number of tokens that you are using.
Let's walk through our two main utility tools below.

1. `OnDemandToolLoader`: This tool turns any existing LlamaIndex data loader (BaseReader class) into a tool that an agent can use. The tool can be called with all the parameters needed to trigger `load_data` from the data loader, along with a natural language query string. During execution, we first load data from the data loader, index it (for instance with a vector store), and then query it 'on-demand'. All three of these steps happen in a single tool call.
2. `LoadAndSearchToolSpec`: The LoadAndSearchToolSpec takes in any existing Tool as input. As a tool spec, it implements `to_tool_list`, and when that function is called, two tools are returned: a loading tool and then a search tool. The load Tool execution would call the underlying Tool, and then index the output (by default with a vector index). The search Tool execution would take in a query string as input and call the underlying index.

> [!TIP]
> You can find toolspecs and utility tools on the LlamaHub

Now that we understand the basics of agents and tools in LlamaIndex, let's see how we can **use LlamaIndex to create configurable and manageable workflows!**

